In [1]:
# Cài đặt Unsloth bản mới nhất và các thư viện anh em đi kèm
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 22.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 70.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
 

In [ ]:
from huggingface_hub import login

# Bỏ cái token của mày vào đây (giữ nguyên cặp dấu ngoặc kép nhé)
login(token="")

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Test thì cứ để 2048 cho nhẹ gánh
dtype = None
load_in_4bit = True # Bật ép cân 4-bit

# 1. Tải model gốc và tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it", # Lấy bản instruct
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Gắn LoRA Adapter (mấy cái r, alpha tao tối ưu sẵn cho mày rồi)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Đỡ tốn VRAM
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.6: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [4]:
from datasets import load_dataset

# THAY ĐƯỜNG DẪN FILE CỦA MÀY VÀO ĐÂY
data_path = "/kaggle/input/datasets/locvu0309/data-5k-line/dataset_5k.jsonl" 
dataset = load_dataset("json", data_files=data_path, split="train")

def format_chat(row):
    # Áp dụng template ChatML chuẩn của Gemma cho cột 'messages'
    row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False)
    return row

dataset = dataset.map(format_chat)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        output_dir = "gemma-3-4b-checkpoints",
        
        # --- THAY ĐỔI ĐỂ CHẠY THẬT Ở ĐÂY ---
        num_train_epochs = 1,     # Cho nó học hết toàn bộ file 5k dòng (1 vòng)
        logging_steps = 10,       # Cứ 10 bước thì in loss một lần cho đỡ rác màn hình
        
        # --- BẢO HIỂM CHỐNG SẬP NGUỒN KAGGLE ---
        save_strategy = "steps", 
        save_steps = 100,         # Cứ cày được 100 bước là lưu 1 lần
        save_total_limit = 2,     # Chỉ giữ 2 bản mới nhất trên máy cho nhẹ
        push_to_hub = True,       # Bật tính năng tự đẩy lên Hugging Face
        hub_model_id = "Loc-Wu-0309/gemma-3-4b-finetune", # Tên kho lưu trên HF của mày
        hub_strategy = "checkpoint",
        # -----------------------------------
        
        optim = "adamw_8bit",
        seed = 3407,
    ),
)

# Lệnh kích hoạt train thật
trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.369660
20,2.136420
30,2.043999
40,1.908443
50,1.805743
60,1.817086
70,1.844464
80,1.785804
90,1.805187
100,1.815861


Unsloth: Restored added_tokens_decoder metadata in gemma-3-4b-checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-4b-checkpoints/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in gemma-3-4b-checkpoints/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-4b-checkpoints.
Unsloth: Restored added_tokens_decoder metadata in gemma-3-4b-checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-4b-checkpoints/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in gemma-3-4b-checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-4b-checkpoints/checkpoint-300.
Unsloth: Restored added_tokens_decoder metadata in gemma-3-4b-checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3-4b-che